# AD preprocessing — corrected execution and missing-code handling

Paths are resolved from `PROJECT_DIR`, independent of the notebook working directory.
CSV exports are saved in `results/preprocessing` and model-input copies in `preprocessed_data`;
figures are saved in `plots/preprocessing`. Existing APOE4 and digit-span transformations are retained.
NACC missing values are handled per variable using the supplied UDS3/genetic dictionaries.

**Endpoint limitation:** the existing NACC mapping `NACCUDSD=4 → AD` actually denotes
all-cause dementia, not a confirmed Alzheimer’s aetiology. That endpoint has not been
redefined in this repair. The original missing-row thresholds, minimum-visit rules,
age-derived time axis, and education/APOE imputation remain study-design assumptions.
The MCI analysis stops before the first dementia record, so later visits cannot re-enter
the risk set. Participants without an observed MCI event are censored at their last retained
pre-dementia visit; this does not implement a competing-risk model.
Corrected input data require model refitting; earlier model results do not describe these new inputs.


In [ ]:
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

from datetime import datetime, timedelta
# warnings.filterwarnings("default")

from pathlib import Path
import os

# Absolute project paths work from either the repository root or code/.
PROJECT_DIR = Path(os.environ.get("SURVIVAL_PROJECT_DIR", "/Users/yihan/Documents/Github/survival_Anlysis_Clustering")).expanduser().resolve()
RAW_DIR = Path(os.environ.get("SURVIVAL_RAW_DIR", str(PROJECT_DIR / "raw_data")))
PREPROCESSED_DIR = Path(os.environ.get("SURVIVAL_PREPROCESSED_DIR", str(PROJECT_DIR / "preprocessed_data")))
RESULTS_DIR = Path(os.environ.get("SURVIVAL_PREPROCESS_RESULTS_DIR", str(PROJECT_DIR / "results" / "preprocessing")))
PLOTS_DIR = Path(os.environ.get("SURVIVAL_PREPROCESS_PLOTS_DIR", str(PROJECT_DIR / "plots" / "preprocessing")))
for directory in (PREPROCESSED_DIR, RESULTS_DIR, PLOTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)
for filename in ("naming_convension.xlsx", "AIBL_2023.xlsx", "investigator_ftldlbd_nacc66.csv"):
    if not (RAW_DIR / filename).is_file():
        raise FileNotFoundError(f"Missing input: {RAW_DIR / filename}. Set SURVIVAL_RAW_DIR to the raw-data folder.")

def export_preprocessed(frame, filename):
    # Keep model input paths compatible; also save the requested result CSV.
    frame.to_csv(PREPROCESSED_DIR / filename, index=False)
    frame.to_csv(RESULTS_DIR / filename, index=False)

def stop_before_first_dementia(frame):
    # A first dementia diagnosis ends MCI ascertainment; later records cannot
    # re-enter the risk set after an AD/dementia row has been dropped.
    parts = []
    for _, group in frame.sort_values(['ID', 'VISIT'], kind='stable').groupby('ID', sort=False):
        hits = np.flatnonzero(group['Classification'].eq('AD').to_numpy())
        parts.append(group.iloc[:hits[0]].copy() if len(hits) else group.copy())
    return pd.concat(parts, ignore_index=True) if parts else frame.iloc[:0].copy()

def apply_by_id(frame, function):
    # Explicit iteration preserves ID columns on pandas 2.x/3.x and avoids
    # ambiguous ID index/column labels introduced by chained groupby.apply.
    parts = [function(group.copy()) for _, group in frame.groupby("ID", sort=False)]
    return pd.concat(parts, ignore_index=True) if parts else frame.iloc[:0].copy()


In [ ]:
naming_key = pd.read_excel(RAW_DIR / 'naming_convension.xlsx',sheet_name='keys')
naming_demo = pd.read_excel(RAW_DIR / 'naming_convension.xlsx',sheet_name='Demographic')
naming_neuro = pd.read_excel(RAW_DIR / 'naming_convension.xlsx',sheet_name='neuropsychological')
naming = pd.concat([naming_key, naming_demo, naming_neuro], ignore_index=True)

## some help function

In [ ]:
def calculate_visit_statistics(df):
    visit_counts = df.groupby('ID').size()
    visit_stats = {
        'average_visits': visit_counts.mean(),
        'min_visits': visit_counts.min(),
        'max_visits': visit_counts.max()
    }
    return visit_stats

In [ ]:
def summarize_classification_progression(df):
    progression_dict = df.groupby('ID')['Classification'].apply(
        lambda x: [v for i, v in enumerate(x) if i == 0 or v != x.iloc[i-1]]
    ).to_dict()

    summary_map = {}
    for id, progression in progression_dict.items():
        progression_tuple = tuple(progression)
        if progression_tuple not in summary_map:
            summary_map[progression_tuple] = []
        summary_map[progression_tuple].append(id)

    for pattern, ids in summary_map.items():
        print(f"Progression Pattern: {pattern} -> RIDs: {ids}")
        print("with", len(ids), 'participants')

    return

def summarize_classification_progression_simple(df):
    progression_dict = df.groupby('ID')['Classification'].apply(
        lambda x: [x.iloc[0], x.iloc[-1]]
    ).to_dict()

    summary_map = {}
    for id, progression in progression_dict.items():
        progression_tuple = tuple(progression)
        if progression_tuple not in summary_map:
            summary_map[progression_tuple] = []
        summary_map[progression_tuple].append(id)

    for pattern, ids in summary_map.items():
        print(f"Progression Pattern: {pattern} -> RIDs: {ids}")
        print("with", len(ids), 'participants')

    return

In [ ]:
def impute_demographic_data(df):
    df["Demographic.ApoE genotype"] = df.groupby("AIBL ID")["Demographic.ApoE genotype"].transform(lambda x: x.ffill().bfill())

    education_mapping = {
        '0-6': (4, 6),
        '7-8': (7, 8),
        '9-12': (9, 12),
        '13-15': (13, 15),
        '15+': (16, 22)
    }

    def impute_years_of_education_exact(row):
        if pd.isna(row['Demographic.Years of Education Exact']) and row['Demographic.Years of Education'] in education_mapping:
            low, high = education_mapping[row['Demographic.Years of Education']]
            return np.random.default_rng(42).integers(low, high + 1) 
        return row['Demographic.Years of Education Exact']

    # Apply imputation function to each row
    df['Demographic.Years of Education Exact'] = df.apply(impute_years_of_education_exact, axis=1)

    return df

In [ ]:
def generate_visit_column(df):
    df = df.sort_values(['ID', 'age'], kind='stable').copy()
    first_age = df.groupby('ID')['age'].transform('first')
    # Preserve the original age-derived integer-month definition.
    df['VISIT'] = ((df['age'] - first_age) * 12).astype(int)
    df['_missing_count'] = df.isna().sum(axis=1)
    df = (df.sort_values(['ID', 'VISIT', '_missing_count', 'age'],
                        ascending=[True, True, True, False], kind='stable')
            .drop_duplicates(['ID', 'VISIT'], keep='first')
            .drop(columns='_missing_count'))
    return df.sort_values(['ID', 'VISIT'], kind='stable').reset_index(drop=True)


## AIBL

In [ ]:
aibl_columns = set(naming['AIBL'].dropna()) - {'Age'}
aibl_columns.update(['AIBL ID', 'Demographic.YearMonthOfBirth', 'Progress Summary.Date of NP assessment',
                     'Demographic.Years of Education', 'Demographic.Years of Education Exact'])
AIBL_raw = pd.read_excel(RAW_DIR / 'AIBL_2023.xlsx', usecols=lambda c: c in aibl_columns)
year_month = AIBL_raw['Demographic.YearMonthOfBirth'].astype('string').str.strip().str.replace(r'\.0$', '', regex=True)
AIBL_raw['Demographic.YearMonthOfBirth'] = pd.to_datetime(year_month + '01', format='%Y%m%d', errors='coerce')
AIBL_raw['Progress Summary.Date of NP assessment'] = pd.to_datetime(
    AIBL_raw['Progress Summary.Date of NP assessment'], dayfirst=True, errors='coerce')
AIBL_raw['Age'] = ((AIBL_raw['Progress Summary.Date of NP assessment'] - AIBL_raw['Demographic.YearMonthOfBirth']).dt.days / 365).round(2)
AIBL_raw = impute_demographic_data(AIBL_raw)


In [ ]:
AIBL_selected = AIBL_raw[naming['AIBL'].dropna().tolist()].copy()
AIBL_selected.replace('NI', np.nan, inplace=True)
AIBL_selected = AIBL_selected.dropna(subset=['Age']).reset_index(drop=True)
AIBL_rename_dict = dict(zip(naming['AIBL'], naming['New_Name']))
AIBL_selected = AIBL_selected.rename(columns=AIBL_rename_dict)
AIBL_selected = generate_visit_column(AIBL_selected)

In [ ]:
baseline = AIBL_selected.groupby('ID', sort=False).head(1).reset_index(drop=True)
education_mean = baseline['edu'].mean()
education_std = baseline['edu'].std()


print(education_mean, education_std)

## NACC

In [ ]:
def add_age_column_NACC(df):
    df = df.copy()
    # Parse calendar fields before cleaning assessment-specific missing codes.
    df['VISITDATE'] = pd.to_datetime(dict(year=df['VISITYR'], month=df['VISITMO'], day=df['VISITDAY']), errors='coerce')
    df['BIRTHDATE'] = pd.to_datetime(dict(year=df['BIRTHYR'], month=df['BIRTHMO'], day=1), errors='coerce')
    df['AGE'] = ((df['VISITDATE'] - df['BIRTHDATE']).dt.total_seconds() / (365.25 * 24 * 60 * 60)).round(2)
    df['AGE'] = df['AGE'].where(df['AGE'].between(18, 120))
    if 'NACCAGE' in df:
        fallback = pd.to_numeric(df['NACCAGE'], errors='coerce')
        df['AGE'] = df['AGE'].fillna(fallback.where(fallback.between(18, 120)))
    return df

# NACC UDS3 RDD: EDUC p55, NACCAGE p56; physical pp125-130;
# CDR, GDS and C1 sections; genetic RDD NACCAPOE p6.
# Valid 8/9 scores and valid 88/99 vital signs MUST NOT be removed globally.
NACC_VALID_RANGES = {
    'EDUC': (0,36), 'SEX': (1,2), 'NACCAPOE': (1,6), 'NACCUDSD': (1,4),
    'NACCBMI': (10,100), 'BPSYS': (70,230), 'BPDIAS': (30,140), 'HRATE': (33,160),
    'CDRSUM': (0,18), 'NACCGDS': (0,15), 'NACCMMSE': (0,30),
    'LOGIMEM': (0,25), 'MEMUNITS': (0,25), 'DIGIF': (0,12), 'DIGIB': (0,12),
    'ANIMALS': (0,77), 'BOSTON': (0,30),
}

def clean_nacc_selected(df, naming):
    df = df.copy()
    audit = []
    ranges = dict(NACC_VALID_RANGES)
    for row in naming.itertuples(index=False):
        if str(row.New_Name).startswith('NPIQ') and pd.notna(row.NACC):
            ranges[row.NACC] = (1,3) if str(row.New_Name).endswith('_Sev') else (0,1)
    for column, (low, high) in ranges.items():
        if column not in df:
            continue
        values = pd.to_numeric(df[column], errors='coerce')
        invalid = values.notna() & ~values.between(low, high)
        audit.append({'variable':column, 'valid_min':low, 'valid_max':high,
                      'nonmissing_input':int(values.notna().sum()), 'masked_out_of_range':int(invalid.sum())})
        df[column] = values.where(values.between(low, high))
    return df, pd.DataFrame(audit)


In [ ]:
nacc_columns = set(naming['NACC'].dropna()) - {'AGE'}
nacc_columns.update(['NACCID','VISITYR','VISITMO','VISITDAY','BIRTHYR','BIRTHMO','NACCAGE'])
NACC_raw = pd.read_csv(RAW_DIR / 'investigator_ftldlbd_nacc66.csv', usecols=lambda c: c in nacc_columns, low_memory=False)
NACC_raw = add_age_column_NACC(NACC_raw)
NACC_raw, missing_code_audit = clean_nacc_selected(NACC_raw, naming)
missing_code_audit.to_csv(RESULTS_DIR / 'NACC_missing_code_audit_AD.csv', index=False)
NACC_selected = NACC_raw[naming['NACC'].dropna().tolist()].copy()
NACC_selected = NACC_selected.dropna(subset=['AGE']).reset_index(drop=True)


In [ ]:
## check for the recalculation of age value
age_difference = abs(NACC_raw['AGE'] - NACC_raw['NACCAGE'])
difference_check = age_difference > 1
rows_with_difference = NACC_raw[difference_check]
print(f"Number of rows with a difference > 1 year: {difference_check.sum()}")
rows_with_difference

In [ ]:
NACC_rename_dict = dict(zip(naming['NACC'], naming['New_Name']))
NACC_selected = NACC_selected.rename(columns=NACC_rename_dict)
NACC_selected = generate_visit_column(NACC_selected)
NACC_selected.head(4)

In [ ]:
baseline = NACC_selected.groupby('ID', sort=False).head(1).reset_index(drop=True)
education_mean = baseline['edu'].mean()
education_std = baseline['edu'].std()


print(education_mean, education_std)

## Check

check between AIBL and NACC, make sure with the same data structure

In [ ]:
NACC_selected['ID'] = NACC_selected['ID'].str.replace('NACC', '').astype(int)
NACC_selected['APOE'] = NACC_selected['APOE'].map({1:'E3/E3' , 2:'E3/E4', 3:'E3/E2', 4:'E4/E4', 5:'E4/E2', 6:'E2/E2'})
NACC_selected['APOE4'] = NACC_selected['APOE'].apply(lambda x: 1 if isinstance(x, str) and 'E4' in x.split('/') else (0 if isinstance(x, str) else None))
# NACC_selected['height'] = NACC_selected['height'] * 2.54 # inches to cm
# NACC_selected['weight'] = NACC_selected['weight'] * 0.453592 # lbs to kg
NACC_selected['Digit_Span_F'] = (NACC_selected['Digit_Span_F'] / 12) * 16 # convert the range from 0-12 to 0-16 be consistent (later in modelling will do normalisation)
NACC_selected['Digit_Span_B'] = (NACC_selected['Digit_Span_B'] / 12) * 14 # convert the range from 0-12 to 0-14 be consistent (later in modelling will do normalisation)

NACC_selected['Classification'] = NACC_selected['Classification'].map({1:'HC' , 3:'MCI', 4:'AD'})
NACC_selected = NACC_selected[NACC_selected['Classification'].isin(['HC', 'MCI', 'AD'])]
NACC_selected.head(4)

In [ ]:
AIBL_selected['sex'] = AIBL_selected['sex'].map({'Female': 2, 'Male': 1})
AIBL_selected['APOE4'] = AIBL_selected['APOE'].apply(lambda x: 1 if isinstance(x, str) and 'E4' in x.split('/') else (0 if isinstance(x, str) else None))

NPIQ_cols_convert_AIBL = ['NPIQ_Delusions', 'NPIQ_Hallucinations', 'NPIQ_Agitation', 'NPIQ_Depression', 'NPIQ_Anxiety', 'NPIQ_Elation', 'NPIQ_Apathy', 'NPIQ_Disinhibition','NPIQ_Irritability', 'NPIQ_Motor', 'NPIQ_Nighttime', 'NPIQ_Appetite']
AIBL_selected[NPIQ_cols_convert_AIBL] = AIBL_selected[NPIQ_cols_convert_AIBL].replace({'Yes': 1, 'No': 0})
NPIQ_cols_numeric_AIBL = ['NPIQ_Hallucinations_Sev', 'NPIQ_Depression_Sev', 'NPIQ_Anxiety_Sev', 'NPIQ_Elation_Sev', 'NPIQ_Apathy_Sev', 'NPIQ_Disinhibition_Sev', 'NPIQ_Irritability_Sev', 'NPIQ_Motor_Sev', 'NPIQ_Nighttime_Sev', 'NPIQ_Appetite_Sev']  
AIBL_selected[NPIQ_cols_numeric_AIBL] = AIBL_selected[NPIQ_cols_numeric_AIBL].apply(pd.to_numeric, errors='coerce')

AIBL_selected['Classification'] = AIBL_selected['Classification'].replace('MCIX', 'MCI')
AIBL_selected = AIBL_selected[AIBL_selected['Classification'].isin(['HC', 'MCI', 'AD'])] # Keep only rows where the 'classification' column contains 'HC', 'MCI', or 'AD'
AIBL_selected.head(4)

## Filter 

In [ ]:
def check_classification(group_df):
    group_df = group_df.sort_values('age', kind='stable')
    hits = np.flatnonzero(group_df['Classification'].eq('AD').to_numpy())
    return group_df.iloc[:hits[0] + 1].copy() if len(hits) else group_df.copy()

def drop_record_num(df, num=1):
    id_counts = df['ID'].value_counts()
    valid_ids = id_counts[id_counts > num].index
    return df[df['ID'].isin(valid_ids)].reset_index(drop=True)

def drop_missing_records(df, threshold=0.6):
    return df[df.isnull().mean(axis=1) <= threshold]

In [ ]:
def remove_after_first_ad(df):
    return apply_by_id(df.sort_values(['ID', 'age'], kind='stable'), check_classification)


In [ ]:
def filter_records(group):
    while len(group) > 1 and (group['VISIT'].iloc[-1] - group['VISIT'].iloc[-2] < 6):
        group = pd.concat([group.iloc[:-2], group.iloc[-1:]])  # Remove second last record
    return group

In [ ]:
print("the shape of DF", NACC_selected.shape, "There are ", len(NACC_selected["ID"].unique()), "unique participants")
NACC_filter = NACC_selected.sort_values(by=['ID', 'VISIT'])
NACC_filter = NACC_filter[NACC_filter.columns[~NACC_filter.columns.str.startswith('NPIQ')]]

NACC_filter = drop_missing_records(NACC_filter, threshold=0)
participants_to_remove = NACC_filter.groupby('ID').first().query("Classification == 'AD'").index
NACC_filter = NACC_filter[~NACC_filter['ID'].isin(participants_to_remove)]
NACC_filter = apply_by_id(NACC_filter, check_classification)
NACC_filter = remove_after_first_ad(NACC_filter)
NACC_filter = drop_record_num(NACC_filter, num=3)
NACC_filter = apply_by_id(NACC_filter, filter_records)
NACC_filter['VISIT'] = NACC_filter.groupby('ID')['VISIT'].transform(lambda x: x - x.min())
print("the shape of DF", NACC_filter.shape, "There are ", len(NACC_filter["ID"].unique()), "unique participants")

In [ ]:
print("the shape of DF", AIBL_selected.shape, "There are ", len(AIBL_selected["ID"].unique()), "unique participants")
AIBL_filter = AIBL_selected.sort_values(by=['ID', 'VISIT'])
AIBL_filter = AIBL_filter[AIBL_filter.columns[~AIBL_filter.columns.str.startswith('NPIQ')]]
AIBL_filter = drop_missing_records(AIBL_filter, threshold=0.3)
participants_to_remove = AIBL_filter.groupby('ID').first().query("Classification == 'AD'").index
AIBL_filter = AIBL_filter[~AIBL_filter['ID'].isin(participants_to_remove)]
AIBL_filter = apply_by_id(AIBL_filter, check_classification)
AIBL_filter = remove_after_first_ad(AIBL_filter)
AIBL_filter = drop_record_num(AIBL_filter,num=3)
AIBL_filter = apply_by_id(AIBL_filter, filter_records)
AIBL_filter['VISIT'] = AIBL_filter.groupby('ID')['VISIT'].transform(lambda x: x - x.min())
print("the shape of DF", AIBL_filter.shape, "There are ", len(AIBL_filter["ID"].unique()), "unique participants")

In [ ]:
calculate_visit_statistics(AIBL_filter)

In [ ]:
(AIBL_filter.isna().mean() * 100).sort_values(ascending=False)

In [ ]:
export_preprocessed(NACC_filter, 'NACC_selected_AD.csv')
export_preprocessed(AIBL_filter, 'AIBL_selected_AD.csv')

In [ ]:
summarize_classification_progression_simple(NACC_filter)

In [ ]:
summarize_classification_progression_simple(AIBL_filter)

In [ ]:
calculate_visit_statistics(AIBL_filter)

In [ ]:
calculate_visit_statistics(NACC_filter)

In [ ]:
def calculate_stats(df, dataset_name):
    df = df.sort_values(by=['ID', 'VISIT'])
    baseline = df.groupby('ID', sort=False).head(1).reset_index(drop=True)

    age_mean = baseline['age'].mean()
    age_std = baseline['age'].std()

    gender_female_count = (baseline['sex'] == 2).sum()
    gender_female_percentage = (gender_female_count / len(baseline)) * 100

    education_mean = baseline['edu'].mean()
    education_std = baseline['edu'].std()

    APOE4_positive_count = (baseline['APOE4'] == 1).sum()
    APOE4_positive_percentage = (APOE4_positive_count / len(baseline)) * 100

    num_visits = df.groupby('ID')['VISIT'].count()
    num_visits_mean = num_visits.mean()
    num_visits_std = num_visits.std()

    follow_up_intervals = df.groupby('ID')['VISIT'].agg(['min', 'max'])
    follow_up_range = f"{follow_up_intervals['max'].min()} - {follow_up_intervals['max'].max()} months"

    last_record_intervals = df.groupby('ID')['VISIT'].agg(lambda x: abs(x.nlargest(2).diff().iloc[-1]))
    follow_up_range_last = f"{last_record_intervals.min()} - {last_record_intervals.max()} months"


    df['AD_converter'] = df['Classification'].eq('AD').astype(int)
    AD_converters = df.groupby('ID')['AD_converter'].max().reset_index()
    AD_conversion_count = AD_converters['AD_converter'].sum()
    AD_conversion_percentage = (AD_conversion_count / AD_converters.shape[0]) * 100

    converters = df[df['ID'].isin(AD_converters[AD_converters['AD_converter'] == 1]['ID'])]
    if not converters.empty:
        time_to_conversion = converters[converters['Classification'] == 'AD'].groupby('ID')['VISIT'].min()
        time_to_conversion_mean = time_to_conversion.mean()
        time_to_conversion_std = time_to_conversion.std()
    else:
        time_to_conversion_mean = None
        time_to_conversion_std = None
    return {
        'Dataset': dataset_name,
        'Age at Baseline': f"{age_mean:.1f} ± {age_std:.1f}",
        'Gender (Female, n, %)': f"{gender_female_count} ({gender_female_percentage:.1f}%)",
        'Education (years, mean ± std)': f"{education_mean:.1f} ± {education_std:.1f}",
        'APOE4 Positive (n, %)': f"{APOE4_positive_count} ({APOE4_positive_percentage:.1f}%)",
        'Visits (mean ± std)': f"{num_visits_mean:.1f} ± {num_visits_std:.1f}",
        'Follow-up Interval': follow_up_range,
        'Follow-up Interval Last': follow_up_range_last,
        'AD Conversion (n, %)': f"{AD_conversion_count} ({AD_conversion_percentage:.1f}%)",
        'Time to AD Conversion (mean ± std)': f"{time_to_conversion_mean:.1f} ± {time_to_conversion_std:.1f}" if time_to_conversion_mean is not None else "N/A"
    }

datasets = [(NACC_filter, "NACC"), (AIBL_filter, "AIBL")]
summary_stats = pd.DataFrame()
for cur_df, cur_df_name in datasets:
    summary_stats = pd.concat([summary_stats, pd.DataFrame([calculate_stats(cur_df, cur_df_name)])], ignore_index=True)
summary_stats

summary_stats.to_csv(RESULTS_DIR / 'cohort_summary_AD.csv', index=False)


## prepare for later survival analysis

In [ ]:
def prepare_for_survival_analysis(df):
    parts = []
    for _, group in df.sort_values(['ID','VISIT'], kind='stable').groupby('ID', sort=False):
        group = group.reset_index(drop=True).copy()
        hits = np.flatnonzero(group['Classification'].eq('AD').to_numpy())
        if len(hits):
            group = group.iloc[:hits[0]+1].copy()
        group['start'] = group['VISIT']
        group['stop'] = group['VISIT'].shift(-1)
        group['event'] = 0
        retained = group.iloc[:-1].copy()
        if len(hits) and not retained.empty:
            retained.loc[retained.index[-1], 'event'] = 1
        parts.append(retained)
    out = pd.concat(parts, ignore_index=True) if parts else df.assign(start=np.nan, stop=np.nan, event=0).iloc[:0]
    out = out.drop(columns='VISIT')
    if not (out['stop'] > out['start']).all():
        raise ValueError('Non-positive survival interval; check visit ordering.')
    if (out.groupby('ID')['event'].sum() > 1).any():
        raise ValueError('More than one first event per participant.')
    return out

def get_participants_with_first_classification_HC(df):
    ordered = df.sort_values(['ID','age'], kind='stable')
    first = ordered.groupby('ID', sort=False).head(1)
    return ordered[ordered.ID.isin(first.loc[first.Classification.eq('HC'), 'ID'])].reset_index(drop=True)

def get_participants_with_first_classification_MCI(df):
    ordered = df.sort_values(['ID','age'], kind='stable')
    first = ordered.groupby('ID', sort=False).head(1)
    return ordered[ordered.ID.isin(first.loc[first.Classification.eq('MCI'), 'ID'])].reset_index(drop=True)


In [ ]:
NACC_survival = prepare_for_survival_analysis(NACC_filter)
NACC_survival = drop_record_num(NACC_survival)
print("There are", NACC_survival['ID'].nunique(), "unique participants selected, with ", round((NACC_survival['event'].sum()/NACC_survival['ID'].nunique())*100, 2), "% convert to AD in NACC")


AIBL_survival = prepare_for_survival_analysis(AIBL_filter)
AIBL_survival= drop_record_num(AIBL_survival)
print("There are", AIBL_survival['ID'].nunique(), "unique participants selected, with ", round((AIBL_survival['event'].sum()/AIBL_survival['ID'].nunique())*100, 2), "% convert to AD in AIBL")

In [ ]:
export_preprocessed(AIBL_survival, 'longitudinal_AIBL_AD.csv')
export_preprocessed(NACC_survival, 'longitudinal_NACC_AD.csv')

In [ ]:
visit_counts = AIBL_survival.groupby('ID').size().reset_index(name='Visit_Count')
AIBL_last_record = AIBL_survival.groupby('ID').last().reset_index()


df_merged = pd.merge(visit_counts, AIBL_last_record[['ID', 'event']], on='ID', how = 'inner')

fig, ax1 = plt.subplots(figsize=(10, 6))
for event, group in df_merged.groupby('event'):
    ax1.hist(group['Visit_Count'], bins=range(1, df_merged['Visit_Count'].max() + 2),
             alpha=0.5, label=event)

ax1.set_xlabel('Number of Visits')
ax1.set_ylabel('Frequency')
ax1.set_title('Number of Visits per Participant (AIBL, AD-progression)')
ax1.legend(title='Event')

# Create a second y-axis for KDE curve
ax2 = ax1.twinx()
for event, group in df_merged.groupby('event'):
    sns.kdeplot(group['Visit_Count'], ax=ax2, label=f'KDE - {event}', linewidth=2)
ax2.set_ylabel('Density')


# Adjust legend placement
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')

fig.tight_layout()
fig.savefig(PLOTS_DIR / 'AD_AIBL_visit_counts.png', dpi=180, bbox_inches='tight')
fig.savefig(PLOTS_DIR / 'AD_AIBL_visit_counts.pdf', bbox_inches='tight')
plt.show()

In [ ]:
visit_counts = NACC_survival.groupby('ID').size().reset_index(name='Visit_Count')
AIBL_last_record = NACC_survival.groupby('ID').last().reset_index()


df_merged = pd.merge(visit_counts, AIBL_last_record[['ID', 'event']], on='ID', how = 'inner')
fig, ax1 = plt.subplots(figsize=(10, 6))
for event, group in df_merged.groupby('event'):
    ax1.hist(group['Visit_Count'], bins=range(1, df_merged['Visit_Count'].max() + 2),
             alpha=0.5, label=event)

ax1.set_xlabel('Number of Visits')
ax1.set_ylabel('Frequency')
ax1.set_title('Number of Visits per Participant (NACC, AD-progression)')
ax1.legend(title='Event')

# Create a second y-axis for KDE curve
ax2 = ax1.twinx()
for event, group in df_merged.groupby('event'):
    sns.kdeplot(group['Visit_Count'], ax=ax2, label=f'KDE - {event}', linewidth=2)
ax2.set_ylabel('Density')


# Adjust legend placement
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')

fig.tight_layout()
fig.savefig(PLOTS_DIR / 'AD_NACC_visit_counts.png', dpi=180, bbox_inches='tight')
fig.savefig(PLOTS_DIR / 'AD_NACC_visit_counts.pdf', bbox_inches='tight')
plt.show()

In [ ]:
last_event = NACC_survival.groupby('ID').last().reset_index()[['ID', 'event']]

# Calculate time duration (max(End) - min(Start))
time_duration = NACC_survival.groupby('ID').agg({'start': 'min', 'stop': 'max'}).reset_index()
time_duration['Time'] = time_duration['stop'] - time_duration['start']

# Merge both
final_df = last_event.merge(time_duration[['ID', 'Time']], on='ID')
final_df['Time'] = final_df['Time'].astype(int)

fig, ax1 = plt.subplots(figsize=(10, 6))
for event, group in final_df.groupby('event'):
    ax1.hist(group['Time'], bins=range(1, final_df['Time'].max() + 2),
             alpha=0.5, label=event)

ax1.set_xlabel('Duration (Months)')
ax1.set_ylabel('Frequency')
ax1.set_title('Total duration per Participant (NACC, AD-progression)')
ax1.legend(title='Event')

# Create a second y-axis for KDE curve
ax2 = ax1.twinx()
for event, group in final_df.groupby('event'):
    sns.kdeplot(group['Time'], ax=ax2, label=f'KDE - {event}', linewidth=2)
ax2.set_ylabel('Density')


# Adjust legend placement
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')

fig.tight_layout()
fig.savefig(PLOTS_DIR / 'AD_NACC_follow_up.png', dpi=180, bbox_inches='tight')
fig.savefig(PLOTS_DIR / 'AD_NACC_follow_up.pdf', bbox_inches='tight')
plt.show()

In [ ]:
last_event = AIBL_survival.groupby('ID').last().reset_index()[['ID', 'event']]

# Calculate time duration (max(End) - min(Start))
time_duration = AIBL_survival.groupby('ID').agg({'start': 'min', 'stop': 'max'}).reset_index()
time_duration['Time'] = time_duration['stop'] - time_duration['start']

# Merge both
final_df = last_event.merge(time_duration[['ID', 'Time']], on='ID')
final_df['Time'] = final_df['Time'].astype(int)

fig, ax1 = plt.subplots(figsize=(10, 6))
for event, group in final_df.groupby('event'):
    ax1.hist(group['Time'], bins=range(1, final_df['Time'].max() + 2),
             alpha=0.5, label=event)

ax1.set_xlabel('Duration (Months)')
ax1.set_ylabel('Frequency')
ax1.set_title('Total duration per Participant (AIBL, AD-progression)')
ax1.legend(title='Event')

# Create a second y-axis for KDE curve
ax2 = ax1.twinx()
for event, group in final_df.groupby('event'):
    sns.kdeplot(group['Time'], ax=ax2, label=f'KDE - {event}', linewidth=2)
ax2.set_ylabel('Density')


# Adjust legend placement
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')

fig.tight_layout()
fig.savefig(PLOTS_DIR / 'AD_AIBL_follow_up.png', dpi=180, bbox_inches='tight')
fig.savefig(PLOTS_DIR / 'AD_AIBL_follow_up.pdf', bbox_inches='tight')
plt.show()

In [ ]:
static_features = ['sex', 'APOE4', 'edu']
dynamic_features = ['age', 'BMI','blood_pressure_systolic', 'blood_pressure_diastolic', 
                    'CDR_SB', 'GDS', 'MMSE', 'memory_recall1', 'memory_recall2', 'Digit_Span_F', 'Digit_Span_B', 'animal', 'BNT',]
